In [ ]:
import os
import random

import h5py
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless; figures are saved to disk
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
TARGET_LENGTH = 128
N_FREQ_CHANNELS = 256
BATCH_SIZE = 64
NUM_WORKERS = 5
FOLD_IDX = 1
L2_NORMALIZE = False   # True = plot the unit-norm embedding that SupCon sees; False = raw embed_head output

HDF5_PATH = "/scratch/gpfs/MLISANTI/ra0438/all_bursts.hdf5"
CATALOG_PATH = "/home/ra0438/chime-catalog-two-analysis/chimefrbcat2.csv"

CHECKPOINT_DIR = "/scratch/gpfs/MLISANTI/ra0438/resnet_ae_checkpoints_cv_holdout"
OUTPUT_DIR = os.path.join(CHECKPOINT_DIR, "embedding_plots")
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
class CHIMEFRBDataset(Dataset):
    def __init__(self, hdf5_path, catalog_path, target_length):
        self.hdf5_path = hdf5_path
        self.dt = 0.009830400085775182 * 1000
        self._h5 = None

        self.target_length = target_length
        cat = pd.read_csv(catalog_path, low_memory=False)
        cat["repeater_name"] = cat["repeater_name"].fillna("").str.strip()
        self.repeater_set = set(
            cat.loc[cat["repeater_name"] != "", "tns_name"].str.strip()
        )
        with h5py.File(hdf5_path, "r") as f:
            self.keys = list(f.keys())
        self.labels = np.array([int(k in self.repeater_set) for k in self.keys], dtype=np.int64)

    @staticmethod
    def _pad_or_crop(wfall, target_length, center_idx):
        n_freq, n_time = wfall.shape
        half = target_length // 2
        start = center_idx - half
        end = start + target_length

        if start >= 0 and end <= n_time:
            return wfall[:, start:end]

        src_start = max(start, 0)
        src_end = min(end, n_time)
        out = np.zeros((n_freq, target_length), dtype=wfall.dtype)
        dst_start = src_start - start
        out[:, dst_start:dst_start + (src_end - src_start)] = wfall[:, src_start:src_end]
        return out

    def _get_file(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.hdf5_path, "r")
        return self._h5

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]

        f = self._get_file()
        wfall = f[key]["wfall_plot"][:]
        extent = np.array(f[key]["extent"])

        wfall = wfall.astype(np.float32)
        std = wfall.std(axis=1, keepdims=True)
        std[std == 0] = 1.0  # avoid divide-by-zero for masked channels
        wfall = (wfall - wfall.mean(axis=1, keepdims=True)) / std

        peak = round(-extent[0] / self.dt)
        wfall = self._pad_or_crop(wfall, self.target_length, peak)
        tensor = torch.from_numpy(np.ascontiguousarray(wfall))
        label = torch.tensor(int(key in self.repeater_set), dtype=torch.long)

        return tensor, label


# ----------------------------------------------------------------------------
# Model (identical to the training script so the checkpoints load)
# ----------------------------------------------------------------------------
class ResBlock(nn.Module):
    """Pre-activation basic residual block (BN-ReLU-Conv x2) with optional stride."""
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)

        self.shortcut = None
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        sc = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(self.drop(F.relu(self.bn2(out))))
        return out + sc


class UpBlock(nn.Module):
    """2x upsample (transposed conv) followed by residual blocks."""
    def __init__(self, in_ch, out_ch, n_blocks, dropout=0.0):
        super().__init__()
        self.up = nn.Sequential(nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
                                nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2, bias=False))
        self.blocks = nn.Sequential(*[ResBlock(out_ch, out_ch, dropout=dropout) for _ in range(n_blocks)])

    def forward(self, x):
        return self.blocks(self.up(x))


class FRBResNetAE(nn.Module):
    def __init__(self, n_freq, n_time, base_channels=32, n_stages=3, enc_blocks=2, dec_blocks=1,
                 max_channels=256, latent_channels=None, embed_dim=64, dropout=0.1, head_hidden=128,
                 mask_ratio=0.0, mask_patch_f=16, mask_patch_t=8):
        super().__init__()
        down = 2 ** n_stages
        assert n_freq % down == 0 and n_time % down == 0, "n_freq/n_time must be divisible by 2**n_stages"
        assert n_freq % mask_patch_f == 0 and n_time % mask_patch_t == 0

        self.n_freq, self.n_time = n_freq, n_time
        self.mask_ratio = mask_ratio
        self.mask_patch_f, self.mask_patch_t = mask_patch_f, mask_patch_t

        chs = [min(base_channels * 2 ** i, max_channels) for i in range(n_stages + 1)]
        latent_channels = latent_channels or chs[-1]

        # ---- encoder ----
        self.stem = nn.Conv2d(1, chs[0], 3, padding=1, bias=False)
        enc = []
        for i in range(n_stages):
            stage = [ResBlock(chs[i], chs[i + 1], stride=2, dropout=dropout)]
            stage += [ResBlock(chs[i + 1], chs[i + 1], dropout=dropout) for _ in range(enc_blocks - 1)]
            enc.append(nn.Sequential(*stage))
        self.enc_stages = nn.Sequential(*enc)
        self.to_latent = nn.Sequential(nn.BatchNorm2d(chs[-1]), nn.ReLU(inplace=True),
                                       nn.Conv2d(chs[-1], latent_channels, 1))

        # ---- decoder ----
        self.from_latent = nn.Conv2d(latent_channels, chs[-1], 1)
        dec = [UpBlock(chs[i + 1], chs[i], dec_blocks, dropout=dropout) for i in reversed(range(n_stages))]
        self.dec_stages = nn.Sequential(*dec)
        self.dec_out = nn.Sequential(nn.BatchNorm2d(chs[0]), nn.ReLU(inplace=True),
                                     nn.Conv2d(chs[0], 1, 3, padding=1))

        # ---- embedding (mean-pooled latent -> vector) ----
        self.embed_head = nn.Sequential(nn.Linear(latent_channels, latent_channels),
                                        nn.LayerNorm(latent_channels),
                                        nn.ReLU(),
                                        nn.Linear(latent_channels, embed_dim))

        # ---- classifier head (on the embedding) ----
        self.cls_head = nn.Sequential(nn.Linear(embed_dim, head_hidden),
                                      nn.LayerNorm(head_hidden),
                                      nn.ReLU(),
                                      nn.Dropout(dropout),
                                      nn.Linear(head_hidden, head_hidden // 2),
                                      nn.ReLU(),
                                      nn.Dropout(dropout),
                                      nn.Linear(head_hidden // 2, 1))

        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    def random_patch_mask(self, x):
        """Returns pixel mask [B, F, T] with 1 = hidden, hiding exactly mask_ratio of the patches."""
        B = x.size(0)
        gf, gt = self.n_freq // self.mask_patch_f, self.n_time // self.mask_patch_t
        n = gf * gt
        n_mask = int(round(n * self.mask_ratio))
        noise = torch.rand(B, n, device=x.device)
        ids = torch.argsort(noise, dim=1)[:, :n_mask]
        m = torch.zeros(B, n, device=x.device)
        m.scatter_(1, ids, 1.0)
        m = m.view(B, gf, gt)
        m = m.repeat_interleave(self.mask_patch_f, dim=1).repeat_interleave(self.mask_patch_t, dim=2)
        return m

    def encode(self, x):
        h = self.stem(x.unsqueeze(1))                # [B, C0, F, T]
        h = self.enc_stages(h)                       # [B, C_n, F/2^n, T/2^n]
        return self.to_latent(h)                     # [B, L, F/2^n, T/2^n]

    def decode(self, z):
        h = self.dec_stages(self.from_latent(z))
        return self.dec_out(h).squeeze(1)            # [B, F, T]

    def embed(self, z):
        return self.embed_head(z.mean(dim=(2, 3)))   # [B, embed_dim]

    def classify(self, e):
        return self.cls_head(e).squeeze(-1)          # [B]

    def forward(self, x, apply_mask=None):
        if apply_mask is None:
            apply_mask = self.training
        mask = None
        x_in = x
        if apply_mask and self.mask_ratio > 0:
            mask = self.random_patch_mask(x)
            x_in = x * (1 - mask)
        z = self.encode(x_in)
        e = self.embed(z)
        return self.classify(e), self.decode(z), mask, e


def build_model_from_params(p):
    mask_patch_f, mask_patch_t = map(int, p["mask_patch"].split("x"))
    return FRBResNetAE(n_freq=N_FREQ_CHANNELS, n_time=TARGET_LENGTH,
                       base_channels=p["base_channels"], n_stages=p["n_stages"],
                       enc_blocks=p["enc_blocks"], dec_blocks=p["dec_blocks"],
                       max_channels=p["max_channels"], latent_channels=p["latent_channels"],
                       embed_dim=p["embed_dim"], dropout=p["dropout"], head_hidden=p["head_hidden"],
                       mask_ratio=p["mask_ratio"], mask_patch_f=mask_patch_f,
                       mask_patch_t=mask_patch_t).to(device)


# ----------------------------------------------------------------------------
# Load checkpoint + data
# ----------------------------------------------------------------------------
dataset = CHIMEFRBDataset(HDF5_PATH, CATALOG_PATH, TARGET_LENGTH)
n_total = len(dataset)
n_rep = int(dataset.labels.sum())
print(f"Dataset: {n_total} | repeaters: {n_rep} ({100 * n_rep / n_total:.1f}%)")

ckpt_path = os.path.join(CHECKPOINT_DIR, f"fold_{FOLD_IDX}.pt")
print(f"Loading checkpoint: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model = build_model_from_params(ckpt["params"])
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

# ----------------------------------------------------------------------------
# Extract embeddings (encoder -> mean-pooled latent -> embed_head), clean input, no masking
# ----------------------------------------------------------------------------
all_emb = []
all_labels = []

with torch.no_grad():
    for wfall, labels in loader:
        e = model.embed(model.encode(wfall.to(device)))   # [B, embed_dim]
        if L2_NORMALIZE:
            e = F.normalize(e, dim=1)
        all_emb.append(e.cpu().numpy())
        all_labels.append(labels.numpy())

all_emb = np.concatenate(all_emb, axis=0)
all_labels = np.concatenate(all_labels, axis=0)
print(f"Extracted embeddings: {all_emb.shape}")

suffix = f"fold{FOLD_IDX}" + ("_l2" if L2_NORMALIZE else "")
np.save(os.path.join(OUTPUT_DIR, f"embeddings_{suffix}.npy"), all_emb)
np.save(os.path.join(OUTPUT_DIR, f"labels_{suffix}.npy"), all_labels)


def scatter_2d(coords, xlabel, ylabel, title, out_png):
    plt.figure(figsize=(8, 6))
    plt.scatter(coords[all_labels == 0, 0], coords[all_labels == 0, 1],
                s=8, alpha=0.5, label="Non-repeater", color="steelblue")
    plt.scatter(coords[all_labels == 1, 0], coords[all_labels == 1, 1],
                s=8, alpha=0.5, label="Repeater", color="crimson")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()
    print(f"Saved plot -> {out_png}")


# ---- PCA ----
pca = PCA(n_components=2)
emb_pca = pca.fit_transform(all_emb)
var_ratio = pca.explained_variance_ratio_
scatter_2d(emb_pca,
           f"PC1 ({var_ratio[0] * 100:.1f}% var)", f"PC2 ({var_ratio[1] * 100:.1f}% var)",
           f"PCA of ResNet embeddings (Fold {FOLD_IDX} model, all {n_total} FRBs)",
           os.path.join(OUTPUT_DIR, f"embedding_pca_{suffix}.png"))

# ---- t-SNE ----
tsne = TSNE(n_components=2, random_state=SEED)
emb_tsne = tsne.fit_transform(all_emb)
scatter_2d(emb_tsne, "t-SNE 1", "t-SNE 2",
           f"t-SNE of ResNet embeddings (Fold {FOLD_IDX} model, all {n_total} FRBs)",
           os.path.join(OUTPUT_DIR, f"embedding_tsne_{suffix}.png"))